# 01 - DVC: Data Version Control

**Goal:** Learn to version datasets with DVC so you always know which data trained which model.

---

## The Problem

Git tracks code. But your data is too big for Git.

```
Git repo:                          NAS / Cloud:
├── training_semantic.py  (50 KB)  ├── spine_dataset/  (50 GB)
├── config.yaml           (2 KB)   └── knee_dataset/   (30 GB)
└── ...                            

Problem: Which version of the data trained model v8?
Answer: Nobody knows.
```

**DVC** solves this by putting tiny pointer files (`.dvc`) in Git that reference the actual data.

## How DVC Works

```
Git repo                          Remote storage (Azure, NAS, local)
┌───────────────────┐             ┌──────────────────────┐
│ training.py       │             │ ab/cd1234...  (hash) │
│ config.yaml       │             │ ef/gh5678...  (hash) │
│ data.dvc ─────────┼── pointer ──│                      │
│ .dvc/config       │             │                      │
└───────────────────┘             └──────────────────────┘
```

| Component | Where | Size | What it does |
|-----------|-------|------|-------------|
| `.dvc` file | Git repo | ~100 bytes | Points to data (hash + size) |
| `.dvc/config` | Git repo | ~200 bytes | Says where remote storage is |
| Actual data | Remote storage | GBs | The real files |

**The intelligence is in Git. The storage is dumb.**

## Setup

Let's create a mini project to learn DVC hands-on.

We'll work in `/tmp/dvc-demo` so we don't mess with our learning repo.

In [1]:
import os
import shutil
import numpy as np
import json

# Clean start
DEMO_DIR = '/tmp/dvc-demo'
if os.path.exists(DEMO_DIR):
    shutil.rmtree(DEMO_DIR)
os.makedirs(DEMO_DIR)
os.chdir(DEMO_DIR)

print(f"Working in: {os.getcwd()}")

Working in: /tmp/dvc-demo


## Step 1: Initialize Git + DVC

DVC lives on top of Git. You always need both.

In [2]:
# Initialize git repo
!git init
!git branch -m main

Initialized empty Git repository in /tmp/dvc-demo/.git/


In [3]:
# Initialize DVC inside the git repo
!dvc init

# See what DVC created
!ls -la .dvc/

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/iterative/dvc>
total 16
drwxr-xr-x 3 root root 4096 Mar  3 11:10 .
drwxr-xr-x 4 root root 4096 Mar  3 11:10 ..
-rw-r--r-- 1 root root   26 Mar  3 11:10 .gitignore
-rw-r--r-- 1 root root    0 Mar  3 11:10 config
drwxr-xr-x 2 root root 4096 Mar  3 11:10 tmp

In [4]:
# DVC adds its own files to git
!git status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .dvc/.gitignore
	new file:   .dvc/config
	new file:   .dvcignore



In [5]:
# Commit the DVC initialization
!git add .
!git commit -m "Initialize DVC"

[main (root-commit) f3bae56] Initialize DVC
 3 files changed, 6 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore


## Step 2: Create a Synthetic Dataset

Let's simulate a medical imaging dataset. In reality, these would be NIfTI files of spine CTs.

We'll create a simple version: random arrays representing "patient scans" with a CSV split file.

In [6]:
def create_dataset(data_dir, num_patients, version_tag="v1"):
    """Create a synthetic dataset mimicking the spine pipeline structure."""
    os.makedirs(f"{data_dir}/images", exist_ok=True)
    os.makedirs(f"{data_dir}/labels", exist_ok=True)
    
    patients = []
    for i in range(num_patients):
        patient_id = f"patient_{i:04d}"
        
        # Simulate a small 3D volume (in reality: ~512x512x300 voxels)
        image = np.random.randn(32, 32, 16).astype(np.float32)
        label = np.random.randint(0, 5, size=(32, 32, 16)).astype(np.uint8)
        
        np.save(f"{data_dir}/images/{patient_id}.npy", image)
        np.save(f"{data_dir}/labels/{patient_id}.npy", label)
        patients.append(patient_id)
    
    # Create metadata
    metadata = {
        "version": version_tag,
        "num_patients": num_patients,
        "patients": patients
    }
    with open(f"{data_dir}/metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    return patients


# Create dataset v1: 20 patients
patients = create_dataset("data/spine_raw", num_patients=20, version_tag="v1")
print(f"Created dataset with {len(patients)} patients")
!ls data/spine_raw/
!ls data/spine_raw/images/ | head -5

Created dataset with 20 patients
images	labels	metadata.json
patient_0000.npy
patient_0001.npy
patient_0002.npy
patient_0003.npy
patient_0004.npy


## Step 3: Track Data with DVC

Now we tell DVC to track this data folder.

In [7]:
# Tell DVC to track the data directory
!dvc add data/spine_raw

⠋ Checking graph                                       core>
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/tmp/dvc-demo/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding data/spine_raw to cache       0/41 [00:00<?,     ?file/s]
                                                                                
!
Checking out /tmp/dvc-demo/data/spine_raw             |0.00 [00:00,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 41.03file/s]

To track the changes with git, run:

	git add data/spine_raw.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true


DVC did two things:

1. Created `data/spine_raw.dvc` — the pointer file
2. Added `data/spine_raw` to `.gitignore` — so Git ignores the actual data

Let's look at the `.dvc` file:

In [8]:
# The .dvc file is tiny - just a pointer
!cat data/spine_raw.dvc

outs:
- md5: a46ddc73b6801a9fcdf5984822691a87.dir
  size: 1643983
  nfiles: 41
  hash: md5
  path: spine_raw


In [9]:
# Git now sees the pointer file, not the data
!git status

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/

nothing added to commit but untracked files present (use "git add" to track)


In [10]:
# Commit: Git tracks the pointer, DVC tracks the data
!git add data/spine_raw.dvc data/.gitignore
!git commit -m "Add spine dataset v1 (20 patients)"

[main eb4d79d] Add spine dataset v1 (20 patients)
 2 files changed, 7 insertions(+)
 create mode 100644 data/.gitignore
 create mode 100644 data/spine_raw.dvc


## Step 4: Set Up a Remote Storage

In production, this would be Azure Blob Storage. For learning, we use a local folder.

```
Production:  dvc remote add -d azure azure://maia-data-bucket
Learning:    dvc remote add -d local /tmp/dvc-storage
```

Same commands, different backend.

In [11]:
# Create local remote storage (simulates Azure Blob Storage)
!mkdir -p /tmp/dvc-storage

# Tell DVC where to store data
!dvc remote add -d local /tmp/dvc-storage

# See the config
!cat .dvc/config

Setting 'local' as a default remote.
[core]
    remote = local
['remote "local"']
    url = /tmp/dvc-storage


In [12]:
# Push data to remote (like git push, but for data)
!dvc push

# See what's stored (hashed files, not named files)
!find /tmp/dvc-storage -type f | head -5

Pushing
!
  0% Querying remote cache|                          |0/1 [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/tmp/dvc-storage/files/md5'| |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/tmp/dvc-demo/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Pushing to local                     0/42 [00:00<?,     ?file/s]
Pushing                                                                         
42 files pushed
/tmp/dvc-storage/files/md5/f6/079bb58774ebaf87598c39f26295b4
/tmp/dvc-storage/files/md5/26/8a8a98d109217ea454ad758c965845
/tmp/dvc-storage/files/md5/21/93f49bea3cf88310cef88fada28368
/tmp/dvc-storage/files/md5/a6/a8c418df622d28a5c462c193cdfba8
/tmp/dvc-storage/files/md5/1d/3e10d6e84e9b0e164301b12fd7b88d


In [13]:
# Commit the remote config
!git add .dvc/config
!git commit -m "Configure DVC remote storage"

[main 8f7ffc9] Configure DVC remote storage
 1 file changed, 4 insertions(+)


## Step 5: Update the Dataset (Versioning)

Now let's add more patients — this is dataset v2.

In [14]:
# Add 10 more patients to the existing dataset
new_patients = []
for i in range(20, 30):  # patients 20-29
    patient_id = f"patient_{i:04d}"
    image = np.random.randn(32, 32, 16).astype(np.float32)
    label = np.random.randint(0, 5, size=(32, 32, 16)).astype(np.uint8)
    np.save(f"data/spine_raw/images/{patient_id}.npy", image)
    np.save(f"data/spine_raw/labels/{patient_id}.npy", label)
    new_patients.append(patient_id)

# Update metadata
with open("data/spine_raw/metadata.json", 'r') as f:
    metadata = json.load(f)
metadata["version"] = "v2"
metadata["num_patients"] = 30
metadata["patients"].extend(new_patients)
with open("data/spine_raw/metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Dataset now has {metadata['num_patients']} patients")

Dataset now has 30 patients


In [15]:
# Tell DVC about the changes
!dvc add data/spine_raw

# The .dvc file changed (new hash)
!cat data/spine_raw.dvc

⠋ Checking graph                                       core>
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/tmp/dvc-demo/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding data/spine_raw to cache       0/21 [00:00<?,     ?file/s]
                                                                                
!
Checking out /tmp/dvc-demo/data/spine_raw             |0.00 [00:00,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 50.80file/s]

To track the changes with git, run:

	git add data/spine_raw.dvc

To enable auto staging, run:

	dvc config core.autostage true
outs:
- md5: 696bd097364656e2446f31513450eb66.dir
  size: 2465943
  nfiles: 61
  hash: md5
  path: spine_raw


In [16]:
# Commit the new version
!git add data/spine_raw.dvc
!git commit -m "Update spine dataset to v2 (30 patients)"

# Push new data to remote
!dvc push

[main 7cd0dfc] Update spine dataset to v2 (30 patients)
 1 file changed, 3 insertions(+), 3 deletions(-)
Pushing
!
  0% Querying remote cache|                          |0/1 [00:00<?,    ?files/s]
                                                                                
!
  0% Querying remote cache|                          |0/1 [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/tmp/dvc-storage/files/md5'| |0/? [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/tmp/dvc-demo/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Pushing to local                     0/22 [00:00<?,     ?file/s]
Pushing                                                                         
22 files pushed


## Step 6: Time Travel

This is the powerful part. Go back to dataset v1:

In [17]:
# See git history
!git log --oneline

7cd0dfc (HEAD -> main) Update spine dataset to v2 (30 patients)
8f7ffc9 Configure DVC remote storage
eb4d79d Add spine dataset v1 (20 patients)
f3bae56 Initialize DVC


In [18]:
# Current state: v2 (30 patients)
!ls data/spine_raw/images/ | wc -l

30


In [19]:
# Go back to v1
!git checkout HEAD~1 -- data/spine_raw.dvc
!dvc checkout

# Now we have v1 (20 patients)
!ls data/spine_raw/images/ | wc -l

Building workspace index                             |66.0 [00:00, 11.3kentry/s]
Comparing indexes                                    |66.0 [00:00, 13.6kentry/s]
Applying changes                                      |1.00 [00:00, 1.00kfile/s]
M       data/spine_raw/
20


In [20]:
# Go back to v2
!git checkout main -- data/spine_raw.dvc
!dvc checkout

# Back to 30 patients
!ls data/spine_raw/images/ | wc -l

Building workspace index                             |46.0 [00:00, 7.08kentry/s]
Comparing indexes                                    |66.0 [00:00, 13.8kentry/s]
Applying changes                                      |21.0 [00:00, 8.47kfile/s]
M       data/spine_raw/
300m


## Step 7: Simulate `dvc pull` (New Team Member)

When someone clones the repo, they get the code and `.dvc` files but NOT the data. They run `dvc pull` to download it.

In [21]:
# Simulate: delete local data (like a fresh clone)
!rm -rf data/spine_raw
!ls data/
print("Data is gone locally.")

spine_raw.dvc
Data is gone locally.


In [22]:
# Pull data from remote (like a new team member would)
!dvc pull

# Data is back
!ls data/spine_raw/images/ | wc -l

Fetching
!
  0% Checking cache in '/tmp/dvc-demo/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
Fetching                                                                        
Building workspace index                             |1.00 [00:00, 1.82kentry/s]
Comparing indexes                                    |66.0 [00:00, 37.3kentry/s]
Applying changes                                      |61.0 [00:00, 5.37kfile/s]
A       data/spine_raw/
1 file added
300m


## Tracking CSV Splits

In your production code, train/val splits are defined by CSV files:

```
# From spine-segmentation-feature-training-scripts/config/spine_semantic_new_v8.yaml
dataset:
  name: 'temp_data_transfer'
  split_name: '101225_s1ok'      # ← a folder with train.csv and val.csv
```

These CSVs define which patients go to training vs validation. **They should also be DVC-tracked** because changing the split = different experiment.

In [23]:
import pandas as pd

# Create train/val splits like the production code
os.makedirs("data/splits/101225_s1ok", exist_ok=True)

all_patients = [f"patient_{i:04d}" for i in range(30)]

# 80/20 split
train_patients = all_patients[:24]
val_patients = all_patients[24:]

# Match production format: columns are 'image' and 'label'
train_df = pd.DataFrame({
    'image': [f"images/{p}.npy" for p in train_patients],
    'label': [f"labels/{p}.npy" for p in train_patients],
})
val_df = pd.DataFrame({
    'image': [f"images/{p}.npy" for p in val_patients],
    'label': [f"labels/{p}.npy" for p in val_patients],
})

train_df.to_csv("data/splits/101225_s1ok/train.csv", index=False)
val_df.to_csv("data/splits/101225_s1ok/val.csv", index=False)

print(f"Train: {len(train_df)} patients")
print(f"Val: {len(val_df)} patients")
train_df.head()

Train: 24 patients
Val: 6 patients


,image,label
0,images/patient_0000.npy,labels/patient_0000.npy
1,images/patient_0001.npy,labels/patient_0001.npy
2,images/patient_0002.npy,labels/patient_0002.npy
3,images/patient_0003.npy,labels/patient_0003.npy
4,images/patient_0004.npy,labels/patient_0004.npy


In [24]:
# Track splits with DVC too
!dvc add data/splits
!git add data/splits.dvc data/.gitignore
!git commit -m "Add train/val splits for experiment 101225"
!dvc push

⠋ Checking graph                                       core>
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/tmp/dvc-demo/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding data/splits to cache           0/2 [00:00<?,     ?file/s]
                                                                                
!
Checking out /tmp/dvc-demo/data/splits                |0.00 [00:00,    ?files/s]
100% Adding...|███████████████████████████████████████|1/1 [00:00, 116.47file/s]

To track the changes with git, run:

	git add data/splits.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
[main 9ce5716] Add train/val splits for experiment 101225
 2 files changed, 7 insertions(+)
 create mode 100644 data/splits.dvc
Pushing
!
  0

## The Data Registry Pattern

For a startup with multiple projects (spine + knee), the standard approach is:

```
maia-data-registry/            ← Dedicated repo (the catalog)
├── spine/
│   ├── raw_v1.dvc             ← Pointer to 200 patients
│   ├── raw_v2.dvc             ← Pointer to 400 patients
│   ├── splits/
│   │   ├── 101225_s1ok.dvc
│   │   └── 020326_new.dvc
│   └── README.md
├── knee/
│   ├── raw_v1.dvc
│   └── README.md
└── .dvc/config                ← Points to Azure Blob Storage

maia-spine-training/           ← Code repo (uses data from registry)
├── src/
├── config/
└── (no .dvc files here)
```

From a code repo, you'd run:
```bash
dvc import maia-data-registry spine/raw_v2
```

This downloads the data and creates a link back to the registry.

## Connection to Your Production Code

Looking at `spine-segmentation-feature-training-scripts/`:

```yaml
# spine_semantic_new_v8.yaml
dataset:
  name: 'temp_data_transfer'       # ← just a folder name, no versioning
  version: 'v1'                     # ← manual, nobody enforces this
  split_name: '101225_s1ok'         # ← CSV files, not version-controlled
```

**What's missing:**

| Current | With DVC |
|---------|----------|
| Data on NAS, no version tracking | Data on Azure, every version tracked |
| `version: 'v1'` is a string, not enforced | Git commit = exact data snapshot |
| CSV splits can change silently | Splits tracked, changes = new commit |
| "Which data trained model v8?" → unknown | `git log` + `dvc diff` → exact answer |
| FDA audit → panic | FDA audit → `git log --all` |

**To integrate DVC into the training pipeline:**
1. `dvc add data/dataset/temp_data_transfer` → track the raw data
2. `dvc add data/data_splits/101225_s1ok` → track the splits
3. Commit `.dvc` files alongside config changes
4. Set up Azure Blob as DVC remote

## Summary

| Command | What it does |
|---------|-------------|
| `dvc init` | Initialize DVC in a git repo |
| `dvc add <path>` | Track a file/folder (creates `.dvc` pointer) |
| `dvc remote add` | Configure where data is stored |
| `dvc push` | Upload data to remote |
| `dvc pull` | Download data from remote |
| `dvc checkout` | Restore data to match current `.dvc` files |
| `dvc diff` | Show what data changed |

**Key takeaway:** Git tracks your code + `.dvc` pointers. DVC tracks your actual data. Together, every commit tells you exactly which code AND which data were used.

**Next:** MLflow for experiment tracking — because versioning data is only half the story. You also need to track what happened during training.